In [6]:
import pandas as pd
import requests
import io
import os

from collections import defaultdict

In [7]:
url_kamus_alay = 'https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv'
response = requests.get(url_kamus_alay)

In [8]:
df_alay = pd.read_csv(io.StringIO(response.text))
detektor_slang = set(df_alay['slang'].tolist())
print(f"{len(detektor_slang)} kata terdaftar sebagai detektor.\n")

4331 kata terdaftar sebagai detektor.



In [9]:
bank_files = [
  'BCAMobile_reviews_cleaned.csv',
  'BRImo_reviews_cleaned.csv',
  'livin_mandiri_reviews_cleaned.csv',
  'Wondr_BNI_reviews_cleaned.csv'
]

slang_data = defaultdict(lambda: {'frekuensi': 0, 'konteks': []})

In [10]:
for file in bank_files:
  path = f'data/processed/{file}'
  if os.path.exists(path):
    df = pd.read_csv(path)
    df = df.dropna(subset=['content_cleaned'])
    
    # Ambil nama bank untuk label konteks (misal: 'BCAMobile')
    bank_name = file.split('_')[0] 
    
    for ulasan in df['content_cleaned']:
      ulasan_str = str(ulasan)
      # Pecah ulasan jadi kata-kata, pakai set() biar satu ulasan gak kehitung dobel 
      # misal nasabah ngetik "yg yg yg"
      kata_unik_di_ulasan = set(ulasan_str.split())
      
      for kata in kata_unik_di_ulasan:
          if kata in detektor_slang:
              slang_data[kata]['frekuensi'] += 1
              
              # Simpan contoh ulasan asli (maksimal 3 ulasan per kata)
              if len(slang_data[kata]['konteks']) < 3:
                  slang_data[kata]['konteks'].append(f"[{bank_name}] {ulasan_str}")
                  
    print(f"Selesai memproses: {file}")
  else:
      print(f"Skip: {file} tidak ditemukan di folder data/processed/")

Selesai memproses: BCAMobile_reviews_cleaned.csv
Selesai memproses: BRImo_reviews_cleaned.csv
Selesai memproses: livin_mandiri_reviews_cleaned.csv
Selesai memproses: Wondr_BNI_reviews_cleaned.csv


In [11]:
hasil_akhir = []
for kata, info in slang_data.items():
    hasil_akhir.append({
        'Singkatan/Slang': kata,
        'Frekuensi': info['frekuensi'],
        'Contoh Ulasan 1': info['konteks'][0] if len(info['konteks']) > 0 else "",
        'Contoh Ulasan 2': info['konteks'][1] if len(info['konteks']) > 1 else "",
        'Contoh Ulasan 3': info['konteks'][2] if len(info['konteks']) > 2 else ""
    })

In [12]:
df_hasil = pd.DataFrame(hasil_akhir)
df_hasil = df_hasil.sort_values(by='Frekuensi', ascending=False)
display(df_hasil.head(10))

,Singkatan/Slang,Frekuensi,Contoh Ulasan 1,Contoh Ulasan 2,Contoh Ulasan 3
6,gak,23883,[BCAMobile] masa gue isi flazz selalu gagal ke...,[BCAMobile] setelah masuk mbanking aktivasi ul...,[BCAMobile] payah jauh sama livin mandiri kiri...
4,ga,20600,[BCAMobile] qris ga di pakai daritadi loading ...,[BCAMobile] d update malah minta masukin norek...,[BCAMobile] akun nya ga bis akembali
1,aja,18153,[BCAMobile] ribet banget ni apk sumpah dikir v...,[BCAMobile] payah jauh sama livin mandiri kiri...,[BCAMobile] apk dongo login aja kaga
22,udah,15884,[BCAMobile] tiap verifikasi wajah knapa slalu ...,[BCAMobile] indikator lampu merah terus mana s...,[BCAMobile] verifikasi e ktp wajah selalu gaga...
10,yg,14415,[BCAMobile] setelah masuk mbanking aktivasi ul...,[BCAMobile] udah restart udah x masuk qris buf...,[BCAMobile] yg terupdate tiap login mesti x si...
14,bca,10570,[BCAMobile] kasik bintang dulu aku new user bc...,[BCAMobile] bca lgi gangguan apa gimana loadin...,[BCAMobile] tiap verifikasi wajah knapa slalu ...
61,gk,9925,[BCAMobile] sekarang bca mobile gk di akses sy...,[BCAMobile] gk tahu eror mulu,[BCAMobile] nomor udah benar kemarin mau pinda...
0,pake,8049,[BCAMobile] ribet banget ni apk sumpah dikir v...,[BCAMobile] aplikasi nya sampah tiba tiba kelu...,[BCAMobile] yaa tiap mau buat m bca selalu stu...
17,gimana,7320,[BCAMobile] bca lgi gangguan apa gimana loadin...,[BCAMobile] kok nggk nerima kode otp nya sejam...,[BCAMobile] gimana tolong habis ganti hp baru ...
29,minta,5501,[BCAMobile] d update malah minta masukin norek...,[BCAMobile] kenapa sama mau masuk aplikasi mau...,[BCAMobile] dipindahkan hp laen diulang dr awa...


In [13]:
nama_file_output = 'data/interim/deteksi_slang_dengan_konteks.csv'
df_hasil.to_csv(nama_file_output, index=False)